In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
MMFF94_Torch vs RDKit(MMFF94) 검증 스크립트
- 입력 positions (torch, requires_grad=True)와 동일 좌표를 RDKit conformer에 복사
- 총 에너지(kcal/mol)와 힘(kcal/mol/Å) 비교
- autograd: F_torch ≈ -∂E/∂x 검증
"""

from rdkit import Chem
from rdkit.Chem import AllChem
import torch
import numpy as np

# --- RDKit grad 버퍼 유틸 (DoubleVector) ---
def _make_rdkit_double_vector(n3: int):
    """
    RDKit 버전에 따라 DoubleVector 경로가 다를 수 있어 try-import로 처리.
    """
    try:
        from rdkit.ForceField import rdForceField as rdFF
        return rdFF.DoubleVector(n3), "rdkit.ForceField.rdForceField"
    except Exception:
        try:
            # 일부 빌드에선 이 경로가 동작할 수 있음
            from rdkit.Chem.rdForceFieldHelpers import DoubleVector  # type: ignore
            return DoubleVector(n3), "rdkit.Chem.rdForceFieldHelpers"
        except Exception:
            # 마지막 폴백: 파이썬 리스트를 받아주는 빌드도 드물게 존재
            return [0.0] * n3, "python.list"

def rdkit_mmff_energy_forces(mol: Chem.Mol, positions: np.ndarray, variant: str = "MMFF94", confId: int = -1):
    """
    RDKit MMFF 에너지/힘 계산
    positions: (N,3) numpy (Å)
    반환: energy(float, kcal/mol), forces(np.ndarray, (N,3), kcal/mol/Å)
    """
    assert positions.ndim == 2 and positions.shape[1] == 3
    n_atoms = mol.GetNumAtoms()

    # 좌표 복사 (RDKit는 double 필요)
    if confId < 0:
        confId = mol.GetConformer().GetId()
    conf = mol.GetConformer(confId)
    for i in range(n_atoms):
        x, y, z = map(float, positions[i])
        conf.SetAtomPosition(i, (x, y, z))

    # MMFF properties & force field
    props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant=variant)
    if props is None:
        raise RuntimeError("MMFF properties 생성 실패 (분자의 원자 타입/전하 문제일 수 있음).")
    ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=confId)
    if ff is None:
        raise RuntimeError("MMFF force field 생성 실패.")

    energy = float(ff.CalcEnergy())

    buf, src = _make_rdkit_double_vector(3 * n_atoms)
    ff.CalcGrad(buf)  # buf를 in-place로 채움

    # buf → (N,3) numpy
    if isinstance(buf, list):
        grad = np.asarray(buf, dtype=np.float64).reshape(n_atoms, 3)
    else:
        # RDKit DoubleVector는 numpy로 캐스팅 가능
        grad = np.asarray(buf, dtype=np.float64).reshape(n_atoms, 3)
    forces = -grad  # RDKit grad는 dE/dx, 힘은 -grad

    return energy, forces

In [2]:

from mmff94_torch import build_forcefield  # 너의 패키지

# --- 분자 준비 (사용자 예시 그대로) ---
mol = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")  # 아스피린
mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, randomSeed=1)

# 좌표 텐서 (N,3), autograd on
conf = mol.GetConformer()
positions = torch.tensor(conf.GetPositions(), dtype=torch.float32, requires_grad=True)

# ----- 토치 FF 구성 -----
ff = build_forcefield(mol, variant="MMFF94", include_hs=True)

# 1) 토치 에너지/힘/그래드 일치 확인
E_torch = ff.energy(positions)  # scalar tensor
F_torch = ff.forces(positions)  # (N,3) tensor
E_torch.backward()
grad_from_autograd = positions.grad  # (N,3)

# 2) RDKit 기준치 계산
E_rdk, F_rdk = rdkit_mmff_energy_forces(
    mol,
    positions.detach().cpu().numpy(),
    variant="MMFF94",
    confId=-1,
)

# 3) 비교 지표
# 에너지 차이 (abs/rel)
E_abs_diff = float(abs(E_torch.item() - E_rdk))
E_rel_diff = float(E_abs_diff / (abs(E_rdk) + 1e-12))

# 힘 차이 (Linf, L2 평균)
F_rdk_t = torch.from_numpy(F_rdk).to(F_torch.dtype)
F_abs = (F_torch - F_rdk_t).abs()
F_linf = float(F_abs.max())
F_l2_mean = float((F_abs.pow(2).sum(-1).sqrt().mean()))

# autograd 검증: F_torch ≈ -grad(E)
autograd_mismatch = (F_torch + grad_from_autograd).abs()
autograd_linf = float(autograd_mismatch.max())
autograd_l2_mean = float(autograd_mismatch.pow(2).sum(-1).sqrt().mean())

# 4) 출력
print("=== MMFF94_Torch vs RDKit(MMFF94) ===")
print(f"Num atoms: {mol.GetNumAtoms()}")
print(f"Energy (Torch) : {E_torch.item():.8f} kcal/mol")
print(f"Energy (RDKit) : {E_rdk:.8f} kcal/mol")
print(f"Energy |Δ|     : {E_abs_diff:.6e} (rel {E_rel_diff:.6e})")
print()
print(f"Forces |Δ|_inf : {F_linf:.6e} kcal/mol/Å")
print(f"Forces mean L2 : {F_l2_mean:.6e} kcal/mol/Å")
print()
print("Autograd check (should be ~0):")
print(f"max |F_torch + grad| : {autograd_linf:.6e}")
print(f"mean L2(F_torch + grad) : {autograd_l2_mean:.6e}")

# 참고: MMFF94s 비교가 필요하면 variant만 바꿔서 한 번 더 돌리면 됨.
# E_rdk_s, F_rdk_s = rdkit_mmff_energy_forces(mol, positions.detach().numpy(), variant="MMFF94s")


=== MMFF94_Torch vs RDKit(MMFF94) ===
Num atoms: 21
Energy (Torch) : 259.25662231 kcal/mol
Energy (RDKit) : 40.45428025 kcal/mol
Energy |Δ|     : 2.188023e+02 (rel 5.408633e+00)

Forces |Δ|_inf : 1.403861e+02 kcal/mol/Å
Forces mean L2 : 4.540623e+01 kcal/mol/Å

Autograd check (should be ~0):
max |F_torch + grad| : 0.000000e+00
mean L2(F_torch + grad) : 0.000000e+00


In [7]:
# =========================================================
# MMFF94 per-term 디버그 유틸 (RDKit vs Torch)
# - RDKit: 각 항목만 ON/OFF해서 에너지 측정
# - Torch: 네 FF 객체 버퍼(토폴로지/파라미터)로 항별 계산
# - in-plane 각도/선형각까지 RDKit 규칙에 맞춤
# =========================================================
from __future__ import annotations
import math
from typing import Dict, Any

import numpy as np
import torch
from rdkit import Chem
from rdkit.Chem import AllChem

# ---------- 상수 (MMFF 정의) ----------
RAD2DEG = 180.0 / math.pi
FC_BOND  = 143.9325
FC_ANGLE = 0.043844
FC_STBN  = 2.51210
CS = -2.0       # Å^-1 (bond cubic)
CB = -0.007     # deg^-1 (angle cubic)
KE = 332.0716
COUL_BUF = 0.05
SCALE14_ELEC = 0.75

# ---------- 공통 지오메트리 ----------
def _norm(v: torch.Tensor, eps=1e-12) -> torch.Tensor:
    return torch.clamp(torch.linalg.norm(v, dim=-1), min=eps)

def _pair_r(x: torch.Tensor, pairs: torch.Tensor) -> torch.Tensor:
    if pairs.numel() == 0:
        return x.new_zeros((0,))
    return _norm(x[pairs[:, 0]] - x[pairs[:, 1]])

def _dihedral(x: torch.Tensor, tors: torch.Tensor) -> torch.Tensor:
    if tors.numel() == 0:
        return x.new_zeros((0,))
    i, j, k, l = tors[:, 0], tors[:, 1], tors[:, 2], tors[:, 3]
    r1, r2, r3, r4 = x[i], x[j], x[k], x[l]
    b0 = r1 - r2
    b1 = r3 - r2
    b2 = r4 - r3
    b1u = b1 / (torch.linalg.norm(b1, dim=1, keepdim=True) + 1e-15)
    v = b0 - (b0 * b1u).sum(-1, keepdim=True) * b1u
    w = b2 - (b2 * b1u).sum(-1, keepdim=True) * b1u
    vu = v / (torch.linalg.norm(v, dim=1, keepdim=True) + 1e-15)
    wu = w / (torch.linalg.norm(w, dim=1, keepdim=True) + 1e-15)
    xcomp = (vu * wu).sum(-1)
    ycomp = (torch.cross(b1u, vu, dim=1) * wu).sum(-1)
    return torch.atan2(ycomp, xcomp)

def _wilson_oop_deg(x: torch.Tensor, imp: torch.Tensor) -> torch.Tensor:
    if imp.numel() == 0:
        return x.new_zeros((0,))
    i, j, k, l = imp[:, 0], imp[:, 1], imp[:, 2], imp[:, 3]
    ri, rj, rk, rl = x[i], x[j], x[k], x[l]
    n = torch.cross(ri - rj, rk - rj, dim=1)
    nh = n / (torch.linalg.norm(n, dim=1, keepdim=True) + 1e-15)
    v = rl - rj
    vh = v / (torch.linalg.norm(v, dim=1, keepdim=True) + 1e-15)
    chi = torch.asin(torch.clamp((nh * vh).sum(-1), -1.0 + 1e-12, 1.0 - 1e-12))
    return chi * RAD2DEG

def _angle_plain(x: torch.Tensor, ang: torch.Tensor) -> torch.Tensor:
    """기본 ∠i-j-k (라디안)."""
    if ang.numel() == 0:
        return x.new_zeros((0,))
    i, j, k = ang[:, 0], ang[:, 1], ang[:, 2]
    v1 = x[i] - x[j]
    v2 = x[k] - x[j]
    cosT = torch.clamp((v1 * v2).sum(-1) / (
        torch.linalg.norm(v1, dim=1) * torch.linalg.norm(v2, dim=1)
    ), -1.0 + 1e-12, 1.0 - 1e-12)
    return torch.acos(cosT)  # rad

def _angle_mmff_deg(
    x: torch.Tensor,
    angles: torch.Tensor,
    ip_mask: torch.Tensor,
    ip_m: torch.Tensor,
) -> torch.Tensor:
    """
    MMFF 각도(도). 기본은 ∠i-j-k.
    단, center j가 trigonal(heavy 3)인 항목은 j를 (i,k,m) 평면으로 투영한 X로 대체해 ∠i-X-k 사용.
    ip_mask: in-plane 적용 여부 (bool), ip_m: 각 항의 third heavy neighbor index (또는 -1)
    """
    if angles.numel() == 0:
        return x.new_zeros((0,))

    i, j, k = angles[:, 0], angles[:, 1], angles[:, 2]
    v1 = x[i] - x[j]
    v2 = x[k] - x[j]
    cosT = torch.clamp((v1 * v2).sum(-1) / (_norm(v1) * _norm(v2)), -1.0 + 1e-12, 1.0 - 1e-12)
    theta = torch.acos(cosT)  # rad

    if ip_mask.any():
        idx = torch.nonzero(ip_mask, as_tuple=False).squeeze(-1)
        if idx.numel() > 0:
            ii, jj, kk, mm = i[idx], j[idx], k[idx], ip_m[idx]
            ri, rj, rk, rm = x[ii], x[jj], x[kk], x[mm]
            n = torch.cross(ri - rm, rk - rm, dim=1)
            n = n / (torch.linalg.norm(n, dim=1, keepdim=True) + 1e-15)
            t = ((rj - rm) * n).sum(-1, keepdim=True)
            rx = rj - t * n
            a = ri - rx
            b = rk - rx
            cos_ip = torch.clamp((a * b).sum(-1) / (_norm(a) * _norm(b)), -1.0 + 1e-12, 1.0 - 1e-12)
            theta_ip = torch.acos(cos_ip)
            theta = theta.clone()
            theta[idx] = theta_ip

    return theta * RAD2DEG

# ---------- RDKit: 항별 에너지 ----------
def rdkit_term_energies(
    mol: Chem.Mol,
    positions: np.ndarray,
    variant: str = "MMFF94",
    confId: int = -1,
) -> Dict[str, float]:
    """
    RDKit MMFF 항별 에너지(kcal/mol) 딕셔너리
    """
    if confId < 0:
        confId = mol.GetConformer().GetId()
    conf = mol.GetConformer(confId)
    for i, (x, y, z) in enumerate(positions.tolist()):
        conf.SetAtomPosition(i, (float(x), float(y), float(z)))

    props = AllChem.MMFFGetMoleculeProperties(mol, mmffVariant=variant)
    if props is None:
        raise RuntimeError("MMFF properties 생성 실패")

    def _calc(terms_on):
        props.SetMMFFBondTerm(False)
        props.SetMMFFAngleTerm(False)
        props.SetMMFFStretchBendTerm(False)
        props.SetMMFFOopTerm(False)
        props.SetMMFFTorsionTerm(False)
        props.SetMMFFVdWTerm(False)
        props.SetMMFFEleTerm(False)
        for name in terms_on:
            getattr(props, f"SetMMFF{name}Term")(True)
        ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=confId)
        if ff is None:
            raise RuntimeError("MMFF force field 생성 실패")
        return float(ff.CalcEnergy())

    out = {
        "bond":         _calc(["Bond"]),
        "angle":        _calc(["Angle"]),
        "stretch_bend": _calc(["StretchBend"]),
        "oop":          _calc(["Oop"]),
        "torsion":      _calc(["Torsion"]),
        "vdw":          _calc(["VdW"]),
        "ele":          _calc(["Ele"]),
    }
    out["total"] = _calc(["Bond", "Angle", "StretchBend", "Oop", "Torsion", "VdW", "Ele"])
    return out

# ---------- Torch: 항별 에너지 ----------
def torch_term_energies(
    ff: Any,                       # forcefield_full_torch.MMFFForceFieldTorch 객체 가정
    positions: torch.Tensor,
) -> Dict[str, torch.Tensor]:
    """
    FF 버퍼를 사용해 항별 에너지 반환(dict of scalar tensors).
    - e_angle: in-plane + 선형각 규칙 반영
    """
    x = positions

    # 버퍼 꺼내기 (이미 같은 dtype/device일 것)
    b, kb, r0 = ff.bonds, ff.kb, ff.r0
    ang, ka, th0, atype = ff.angles, ff.ka, ff.theta0_deg, ff.angle_type
    ip_mask, ip_m = ff.angle_ip_mask, ff.angle_ip_m
    st, k1, k2 = ff.stbn, ff.kba_ijk, ff.kba_kji
    r0ij, r0kj, th0_sb = ff.sb_r0_ij, ff.sb_r0_kj, ff.sb_theta0_deg
    imp, koop = ff.impropers, ff.koop
    tors, V1, V2, V3 = ff.torsions, ff.V1, ff.V2, ff.V3
    nb, is14, Rstar, eps, qi = ff.nb_pairs, ff.is14, ff.Rstar, ff.eps, ff.qi

    # Bond
    if b.numel():
        r = _pair_r(x, b)
        dr = r - r0
        Eb = (FC_BOND * 0.5 * kb * dr * dr * (1.0 + CS * dr + (7.0/12.0) * (CS**2) * (dr**2))).sum()
    else:
        Eb = x.new_zeros(())

    # Angle (in-plane + linear)
    if ang.numel():
        th = _angle_plain(x, ang)           # rad
        th_deg = th * RAD2DEG
        lin = (atype == 1)
        nlin = ~lin
        Eang = x.new_zeros(())
        if lin.any():
            Eang = Eang + (143.9325 * ka[lin] * (1.0 + torch.cos(th[lin]))).sum()
        if nlin.any():
            dth = th_deg[nlin] - th0[nlin]
            Eang = Eang + (0.5 * 0.043844 * ka[nlin] * dth * dth * (1.0 + (-0.007) * dth)).sum()
    else:
        Eang = x.new_zeros(())

    # Stretch–Bend
    if st.numel():
        i, j, k = st[:, 0], st[:, 1], st[:, 2]
        rij = _pair_r(x, torch.stack([i, j], dim=1))
        rkj = _pair_r(x, torch.stack([k, j], dim=1))
        Drij = rij - r0ij
        Drkj = rkj - r0kj
        th_deg_sb = _angle_mmff_deg(x, st, ip_mask.new_zeros(st.shape[0], dtype=torch.bool), ip_m.new_zeros(st.shape[0]))
        DT = th_deg_sb - th0_sb
        Esb = (FC_STBN * (k1 * Drij + k2 * Drkj) * DT).sum()
    else:
        Esb = x.new_zeros(())

    # OOP
    if imp.numel():
        chi = _wilson_oop_deg(x, imp)
        Eoop = (0.5 * FC_ANGLE * koop * chi * chi).sum()
    else:
        Eoop = x.new_zeros(())

    # Torsion
    if tors.numel():
        w = _dihedral(x, tors)
        Etor = (0.5 * (V1 * (1.0 + torch.cos(w)) + V2 * (1.0 - torch.cos(2.0 * w)) + V3 * (1.0 + torch.cos(3.0 * w)))).sum()
    else:
        Etor = x.new_zeros(())

    # vdW (Buffered 14-7) — 1–4 스케일 일반적으로 없음
    if nb.numel():
        r = _pair_r(x, nb)
        R = Rstar
        t1 = ((1.07 * R) / (r + 0.07 * R)) ** 7
        R7 = R ** 7
        t2 = (1.12 * R7) / (r**7 + 0.12 * R7) - 2.0
        Evdw = (eps * t1 * t2).sum()
    else:
        Evdw = x.new_zeros(())

    # Electrostatics (buffered Coulomb), 1–4 0.75
    if nb.numel():
        r = _pair_r(x, nb)
        qij = qi[nb[:, 0]] * qi[nb[:, 1]]
        base = KE * qij / (r + COUL_BUF)
        scale = torch.where(is14, torch.tensor(SCALE14_ELEC, dtype=base.dtype, device=base.device),
                            torch.tensor(1.0, dtype=base.dtype, device=base.device))
        Eele = (scale * base).sum()
    else:
        Eele = x.new_zeros(())

    return {
        "bond":         Eb.detach(),
        "angle":        Eang.detach(),
        "stretch_bend": Esb.detach(),
        "oop":          Eoop.detach(),
        "torsion":      Etor.detach(),
        "vdw":          Evdw.detach(),
        "ele":          Eele.detach(),
        "total":        (Eb + Eang + Esb + Eoop + Etor + Evdw + Eele).detach(),
    }

# ---------- 프린트 테이블 ----------
def compare_terms_table(
    mol: Chem.Mol,
    positions_torch: torch.Tensor,
    ff: Any,
    variant: str = "MMFF94",
) -> None:
    pos_np = positions_torch.detach().cpu().numpy()
    rd = rdkit_term_energies(mol, pos_np, variant=variant)
    th = {k: float(v.cpu().item()) for k, v in torch_term_energies(ff, positions_torch).items()}

    keys = ["bond", "angle", "stretch_bend", "oop", "torsion", "vdw", "ele", "total"]
    print("\n--- Per-term energy (kcal/mol) ---")
    print(f"{'term':<14}{'RDKit':>16}{'Torch':>16}  diff")
    for k in keys:
        r = rd[k]
        t = th[k]
        print(f"{k:<14}{r:>16.8f}{t:>16.8f}  {t - r:+.8f}")


In [8]:
compare_terms_table(mol, positions, ff, variant="MMFF94")


--- Per-term energy (kcal/mol) ---
term                     RDKit           Torch  diff
bond                5.00563287      5.00563097  -0.00000190
angle              19.44503758    238.24739075  +218.80235317
stretch_bend       -0.07239294     -0.07239437  -0.00000143
oop                 0.00000000      0.00000000  -0.00000000
torsion             8.66044671      8.66044617  -0.00000055
vdw                23.79040004     23.79041100  +0.00001096
ele               -16.37484401    -16.37486458  -0.00002057
total              40.45428025    259.25662231  +218.80234206
